## 1. Setup & Dependencies

In [ ]:
# Install training dependencies
!pip install -q trl>=0.28.0 peft>=0.18.0 transformers>=4.40.0 datasets accelerate bitsandbytes

In [ ]:
import torch
import json
from pathlib import Path

# Verify GPU
assert torch.cuda.is_available(), "No GPU detected — switch to a GPU runtime"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 2. Upload Training Data

Upload `constitutional_pairs.jsonl` from your local machine.
PyCharm's Colab integration can auto-upload missing files,
or you can manually upload via the cell below.

In [ ]:
# Option A: Upload from local machine (run this cell)
from google.colab import files
import os

DATA_PATH = "constitutional_pairs.jsonl"

if not os.path.exists(DATA_PATH):
    print("Upload constitutional_pairs.jsonl:")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

# Verify
with open(DATA_PATH) as f:
    lines = [l for l in f if l.strip()]
print(f"Loaded {len(lines)} preference pairs from {DATA_PATH}")

## 3. Configuration

GPU-optimized hyperparameters. Adjust based on your GPU memory.

In [ ]:
# ── Model ──────────────────────────────────────────────────
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# ── DPO Hyperparameters ────────────────────────────────────
DPO_BETA = 0.1           # KL regularization strength
LEARNING_RATE = 5e-6     # 10x higher than CPU run (GPU allows faster convergence)
NUM_EPOCHS = 3           # 3 epochs (CPU run did 1)
BATCH_SIZE = 4           # per-device batch size (A100: 4-8, V100/T4: 2)
GRAD_ACCUM = 4           # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
MAX_LENGTH = 1024        # max tokens per sequence
MAX_PROMPT_LENGTH = 512  # max tokens for prompt
EVAL_SPLIT = 0.1         # 10% for evaluation
WARMUP_RATIO = 0.1       # linear warmup for 10% of steps

# ── LoRA ───────────────────────────────────────────────────
LORA_RANK = 32           # doubled from CPU run (GPU can handle it)
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]  # all attention projections

# ── Output ─────────────────────────────────────────────────
OUTPUT_DIR = "checkpoints/dpo-gpu"

# Auto-adjust batch size for smaller GPUs
if gpu_mem < 40:  # V100 (16GB) or T4 (16GB)
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    print(f"Adjusted for {gpu_name}: batch_size=1, grad_accum=8")
elif gpu_mem < 60:  # A10G (24GB)
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    print(f"Adjusted for {gpu_name}: batch_size=2, grad_accum=4")
else:  # A100 (40/80GB)
    print(f"Full config for {gpu_name}: batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

## 4. Load Data

In [ ]:
from datasets import Dataset

# Parse JSONL
records = []
with open(DATA_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        records.append({
            "prompt": rec["instruction"],
            "chosen": rec["chosen"],
            "rejected": rec["rejected"],
        })

dataset = Dataset.from_dict({
    "prompt": [r["prompt"] for r in records],
    "chosen": [r["chosen"] for r in records],
    "rejected": [r["rejected"] for r in records],
})

# Train/eval split
split = dataset.train_test_split(test_size=EVAL_SPLIT, seed=42)
train_ds = split["train"]
eval_ds = split["test"]

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"Sample prompt length: {len(train_ds[0]['prompt'])} chars")
print(f"Sample chosen length: {len(train_ds[0]['chosen'])} chars")

## 5. Load Model & Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2" if gpu_mem >= 40 else "eager",
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params / 1e9:.1f}B parameters")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 6. Configure LoRA & DPO Trainer

In [ ]:
from peft import LoraConfig, TaskType
from trl import DPOConfig, DPOTrainer

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=DPO_BETA,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    warmup_ratio=WARMUP_RATIO,
    bf16=True,                        # GPU-accelerated bf16
    gradient_checkpointing=True,       # saves VRAM at cost of ~20% speed
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    logging_steps=1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Steps per epoch: {len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"Total steps: {len(train_ds) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS}")

## 7. Train

In [ ]:
import time

print("Starting DPO training...")
start = time.time()

result = trainer.train()

elapsed = time.time() - start
print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final loss: {result.training_loss:.4f}")
print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 8. Evaluate & Save

In [ ]:
# Run evaluation
eval_results = trainer.evaluate()
print("Evaluation results:")
for k, v in sorted(eval_results.items()):
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Save the final LoRA adapter
SAVE_PATH = "checkpoints/dpo-gpu-final"
trainer.save_model(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

# Show adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH)
    if os.path.isfile(os.path.join(SAVE_PATH, f))
)
print(f"Adapter size: {total_size / 1e6:.1f} MB")

## 9. Quick Inference Test

Generate a sample analysis to verify the fine-tuned model works.

In [ ]:
from peft import PeftModel

# Load base model + LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
finetuned = PeftModel.from_pretrained(base_model, SAVE_PATH)
finetuned.eval()

# Test prompt from training data
test_prompt = train_ds[0]["prompt"][:500]

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = finetuned.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
    )
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("FINE-TUNED MODEL RESPONSE:")
print("=" * 60)
print(response[:1000])

## 10. Download Adapter

Download the trained LoRA adapter back to your local machine.

In [ ]:
# Zip and download
import shutil

archive_name = "dpo-gpu-final"
shutil.make_archive(archive_name, "zip", SAVE_PATH)
print(f"Created {archive_name}.zip")

# Download to local machine
from google.colab import files
files.download(f"{archive_name}.zip")
print("Download started — check your browser downloads")